# Lenguajes y Autómatas II (SCD-1016)
## Sesión 19 · Estructuras (struct) y Offsets de Miembros

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Dónde queda cada campo de una estructura dentro de su bloque de memoria?
>
> **Objetivo de la sesión:** Calcular los offsets de los miembros de una estructura aplicando alineación y padding, y obtener el tamaño total del tipo.

**Entregable:** tus funciones offset_campo() y tamano_struct() en artesano_sesion19_c.c, la salida del harness y la bitácora en auditorias/auditoria_sesion19.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)

> ¿Dudas de C? Consulta la [guía de C](https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2/blob/main/docs/guia-c.md) antes de empezar.


## 1. ¿Qué vas a construir?

Recibes una lista ordenada de campos con su tipo; offset_campo() devuelve la posición del campo pedido y tamano_struct() el total con padding final. El harness compara contra la disposición real de la arquitectura.

Al final de la sesión tendrás: **Un calculador de offsets y tamaños de struct que respeta alineación y padding, verificado con cuatro disposiciones distintas.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 19](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion19.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion19.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion19.h"

int offset_campo(const Campo *campos, int total, const char *nombre);
int tamano_struct(const Campo *campos, int total);
```

- Cada campo se alinea a su propio ancho antes de asignarle offset.
- El tamaño total se redondea al múltiplo del campo más grande.
- Un campo inexistente, una lista NULL o un tipo desconocido devuelven -1.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion19_c.c` junto con `artesano_sesion19_c.c`
> (o `ia_sesion19_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `struct char e int: b` | 4 |
| 2 | `struct de tres campos: c` | 8 |
| 3 | `tamano total con padding` | 12 |
| 4 | `double seguido de char` | 16 |
| 5 | `miembro inexistente` | -1 |
| 6 | `struct sin campos` | 0 |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 19 - Offsets de miembros y tamano de struct ===
Caso                               esperado:              obtenido:              resultado
[ 1] struct char e int: b               esperado: 4            obtenido: 4            OK
[ 2] struct de tres campos: c           esperado: 8            obtenido: 8            OK
[ 3] tamano total con padding           esperado: 12           obtenido: 12           OK
[ 4] double seguido de char             esperado: 16           obtenido: 16           OK
[ 5] miembro inexistente                esperado: -1           obtenido: -1           OK
[ 6] struct sin campos                  esperado: 0            obtenido: 0            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion19.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] Cada offset coincide con la posición real del campo después de aplicar el relleno.
- [ ] El tamaño total incluye el padding final hasta el múltiplo de la alineación mayor.
- [ ] Un miembro inexistente devuelve -1 y una estructura sin campos devuelve 0.


## Paso 0 · Entorno


In [1]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


✅ Entorno listo:
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion19_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [2]:
%%writefile api_sesion19.h
/* api_sesion19.h - contrato de la Sesion 19
 * Offsets de miembros y tamaño de struct
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION19_H
#define API_SESION19_H

typedef struct Campo {
    const char *nombre;
    const char *tipo;
} Campo;

int offset_campo(const Campo *campos, int total, const char *nombre);
int tamano_struct(const Campo *campos, int total);

#endif /* API_SESION19_H */


Writing api_sesion19.h


In [3]:
%%writefile artesano_sesion19_c.c
/* artesano_sesion19_c.c - Fase 1: El Artesano (sin IA)
 * Offsets de miembros y tamaño de struct.
 * No lleva main(), el main es del harness.
 */
#include "api_sesion19.h"
#include <stdio.h>
#include <string.h>

// bytes de cada tipo, -1 si no existe
static int ancho(const char *tipo) {
    if (tipo == NULL) return -1;
    if (strcmp(tipo, "char") == 0)    return 1;
    if (strcmp(tipo, "short") == 0)   return 2;
    if (strcmp(tipo, "int") == 0)     return 4;
    if (strcmp(tipo, "float") == 0)   return 4;
    if (strcmp(tipo, "double") == 0)  return 8;
    if (strcmp(tipo, "puntero") == 0) return 8;
    return -1;
}

// sube x al siguiente multiplo de a
static int alinear(int x, int a) {
    if (x % a != 0) x = x + (a - x % a);
    return x;
}

int offset_campo(const Campo *campos, int total, const char *nombre) {
    if (campos == NULL || nombre == NULL || total < 0) return -1;

    int cursor = 0;
    for (int i = 0; i < total; i++) {
        int a = ancho(campos[i].tipo);
        if (a == -1) return -1; // tipo raro

        cursor = alinear(cursor, a); // primero lo acomodo
        if (strcmp(campos[i].nombre, nombre) == 0) return cursor;
        cursor = cursor + a;         // y luego avanzo
    }
    return -1; // no hay campo con ese nombre
}

int tamano_struct(const Campo *campos, int total) {
    if (campos == NULL || total < 0) return -1;
    if (total == 0) return 0; // struct vacio

    int cursor = 0;
    int mayor = 1; // el campo mas grande
    for (int i = 0; i < total; i++) {
        int a = ancho(campos[i].tipo);
        if (a == -1) return -1;

        cursor = alinear(cursor, a) + a;
        if (a > mayor) mayor = a;
    }
    // padding final: al multiplo del campo mas grande
    return alinear(cursor, mayor);
}

Writing artesano_sesion19_c.c


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion19_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


### Fase 2 · El Científico — Contrato C-R-E-O

**C - Contexto:**
En Lenguajes y Autómatas II estamos viendo cómo se acomodan los campos de un `struct` en memoria. No se pegan uno tras otro nada más: cada campo tiene que empezar en un múltiplo de su tamaño, así que el compilador mete bytes de relleno (padding) entre campos y también al final.
La función recibe una lista ordenada de campos (`nombre` y `tipo`) y cuántos son.
Es en C (C11) y se compila con `-Wall -Wextra -Werror`, sin warnings. Solo librerías estándar y sin `main()`, porque el harness ya trae uno. El struct `Campo` y las firmas están en `api_sesion19.h` y no se cambian.

**R - Rol:**
Que la IA actúe como ingeniero de software de sistemas y compiladores que sabe de alineación, padding y disposición de structs en memoria.

**E - Especificación:**
Anchos de tipo: `char=1`, `short=2`, `int=4`, `float=4`, `double=8`, `puntero=8`.

1. `int offset_campo(const Campo *campos, int total, const char *nombre);`
   - Recorre los campos en orden. Antes de asignarle posición a cada campo, sube el cursor al siguiente múltiplo de su ancho.
   - Si el campo es el que busco, regresa ese offset. Si no, avanza el cursor `ancho` bytes.
   - Regresa -1 si el campo no existe, si `campos` o `nombre` son NULL o si hay un tipo desconocido.

2. `int tamano_struct(const Campo *campos, int total);`
   - Igual que arriba, suma cada campo ya alineado.
   - Al final redondea el total al múltiplo del campo **más grande** de todo el struct (no del último).
   - `total == 0` regresa 0.
   - `campos` NULL o un tipo desconocido regresa -1.

Casos que debe pasar:
| Caso | Esperado |
| :--- | :--- |
| {char a, int b} → offset de b | 4 |
| {char a, int b, short c} → offset de c | 8 |
| {char a, int b, short c} → tamaño | 12 |
| {double a, char b} → tamaño | 16 |
| campo "z" que no existe | -1 |
| struct con 0 campos | 0 |

**O - Output:**
Solo el código de `ia_sesion19_c.c`, corto, sin `main()` y sin explicación larga. Al final una línea con la complejidad.

**Lo que revisé antes de compilar:**
Que no cambiara las firmas, que no trajera `main()` y que el padding final usara el campo más grande y no el último. También comparé los resultados contra `sizeof` y `offsetof` de structs reales.

In [4]:
%%writefile ia_sesion19_c.c
/* ia_sesion19_c.c - Fase 2: El Cientifico
 * Codigo que me dio la IA (misma firma, sin main).
 */
#include "api_sesion19.h"
#include <string.h>
#include <stddef.h>

static int ancho(const char *t) {
    const char *n[] = {"char", "short", "int", "float", "double", "puntero"};
    const int   w[] = { 1,      2,       4,     4,       8,        8 };
    if (!t) return -1;
    for (int i = 0; i < 6; i++) {
        if (strcmp(t, n[i]) == 0) return w[i];
    }
    return -1;
}

// redondea x hacia arriba al multiplo de a
static int redondear(int x, int a) {
    return ((x + a - 1) / a) * a;
}

int offset_campo(const Campo *campos, int total, const char *nombre) {
    if (!campos || !nombre || total < 0) return -1;
    int pos = 0;
    for (int i = 0; i < total; i++) {
        int w = ancho(campos[i].tipo);
        if (w < 0) return -1;
        pos = redondear(pos, w);
        if (campos[i].nombre && strcmp(campos[i].nombre, nombre) == 0) return pos;
        pos += w;
    }
    return -1;
}

int tamano_struct(const Campo *campos, int total) {
    if (!campos || total < 0) return -1;
    if (total == 0) return 0;
    int pos = 0, max = 1;
    for (int i = 0; i < total; i++) {
        int w = ancho(campos[i].tipo);
        if (w < 0) return -1;
        pos = redondear(pos, w) + w;
        if (w > max) max = w;   // alineacion MAYOR, no la del ultimo
    }
    return redondear(pos, max);
}

// Complejidad: offset_campo O(n), tamano_struct O(n), n = numero de campos


Writing ia_sesion19_c.c


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [5]:
%%writefile harness_sesion19_c.c
/* harness_sesion19_c.c - auditoria automatica de la Sesion 19
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion19_c.c artesano_sesion19_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion19.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 19 - Offsets de miembros y tamano de struct ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { Campo a[] = {{"a", "char"}, {"b", "int"}}; int r = offset_campo(a, 2, "b"); verificar_int("struct char e int: b", (long)(4), (long)(r)); }
    { Campo b[] = {{"a", "char"}, {"b", "int"}, {"c", "short"}}; int r = offset_campo(b, 3, "c"); verificar_int("struct de tres campos: c", (long)(8), (long)(r)); }
    { Campo b[] = {{"a", "char"}, {"b", "int"}, {"c", "short"}}; int r = tamano_struct(b, 3); verificar_int("tamano total con padding", (long)(12), (long)(r)); }
    { Campo d[] = {{"a", "double"}, {"b", "char"}}; int r = tamano_struct(d, 2); verificar_int("double seguido de char", (long)(16), (long)(r)); }
    { Campo b[] = {{"a", "char"}, {"b", "int"}, {"c", "short"}}; int r = offset_campo(b, 3, "z"); verificar_int("miembro inexistente", (long)(-1), (long)(r)); }
    { Campo b[] = {{"a", "char"}}; int r = tamano_struct(b, 0); verificar_int("struct sin campos", (long)(0), (long)(r)); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion19.md" "\n");
    return 0;
}


Writing harness_sesion19_c.c


In [6]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion19_c.c artesano_sesion19_c.c -o test_artesano && ./test_artesano

=== AUDITORIA SESION 19 - Offsets de miembros y tamano de struct ===
Caso                               esperado:              obtenido:              resultado
[ 1] struct char e int: b               esperado: 4            obtenido: 4            OK
[ 2] struct de tres campos: c           esperado: 8            obtenido: 8            OK
[ 3] tamano total con padding           esperado: 12           obtenido: 12           OK
[ 4] double seguido de char             esperado: 16           obtenido: 16           OK
[ 5] miembro inexistente                esperado: -1           obtenido: -1           OK
[ 6] struct sin campos                  esperado: 0            obtenido: 0            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion19.md


In [7]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion19_c.c ia_sesion19_c.c -o test_ia && ./test_ia

=== AUDITORIA SESION 19 - Offsets de miembros y tamano de struct ===
Caso                               esperado:              obtenido:              resultado
[ 1] struct char e int: b               esperado: 4            obtenido: 4            OK
[ 2] struct de tres campos: c           esperado: 8            obtenido: 8            OK
[ 3] tamano total con padding           esperado: 12           obtenido: 12           OK
[ 4] double seguido de char             esperado: 16           obtenido: 16           OK
[ 5] miembro inexistente                esperado: -1           obtenido: -1           OK
[ 6] struct sin campos                  esperado: 0            obtenido: 0            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion19.md


## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion19.md` a `auditorias/auditoria_sesion19.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
